# Task 12 — Active Data Acquisition (world-model-accuracy checkpoint)

Runs on Kaggle because this program (7 comparators × 3 tasks, one of which trains a full imagined actor-critic) is too large for the CPU-only MacBook Air this project otherwise runs on.

**Before running:** Add the Kaggle Dataset built from the `kaggle/task12/` folder of the project repo as an **Input** to this notebook (`+ Add Input` → Datasets → upload that folder as a new dataset). It contains: `src/` (world model, RSSM, actor-critic, env wrappers — this project's own code, not reinstalled from anywhere), `checkpoints/` (the three frozen base models), `canonical_null/` (the 50 fixed random directions from the Task 11 extension), and `task12_program.py` (the full implementation). Rename `INPUT_ROOT` below to match your dataset's slug if different.

This notebook is a thin runner around `task12_program.py` — all the logic (and its docstring, which explains every design decision) lives there, not here, so it can be smoke-tested locally before every Kaggle run.

**Resumability:** every (task, comparator) cell's result is written to `/kaggle/working/task12_results/<task>/<name>.json` the moment it finishes. If the session is interrupted (Kaggle's 12h limit, a restart), re-running this notebook skips every cell whose JSON already exists and only computes what's missing — nothing is repeated, nothing is lost.

**Scope:** the world-model-accuracy checkpoint only, per the design doc's own sequencing (`task_12_design_decisions.md` in the project repo) — downstream-return evaluation is a separate, later notebook, gated on these results looking worth the extra Task-6 dependency.

In [ ]:
import sys, os

INPUT_ROOT = '/kaggle/input/task12-signal-check-inputs'  # <-- rename to your dataset's slug
assert os.path.isdir(INPUT_ROOT), f"Input dataset not found at {INPUT_ROOT} -- add it via '+ Add Input' first"
sys.path.insert(0, INPUT_ROOT)

os.environ['TASK12_OUT'] = '/kaggle/working/task12_results'
os.makedirs(os.environ['TASK12_OUT'], exist_ok=True)

try:
    from dm_control import suite  # noqa
except ImportError:
    !pip install -q dm_control
    from dm_control import suite  # noqa

print('dm_control OK')

In [ ]:
# task12_program.py resolves its own checkpoints/canonical_null paths relative
# to its own file location, which is INPUT_ROOT once imported from there.
import importlib
sys.path.insert(0, INPUT_ROOT)
import task12_program as P
importlib.reload(P)

# fixed constants, confirm before running at full scale (see task_12_design_decisions.md):
print('N_BUDGET       =', P.N_BUDGET)
print('FINETUNE_STEPS =', P.FINETUNE_STEPS)
print('N_EVAL_TRAJ    =', P.N_EVAL_TRAJ)
print('N_NULL         =', P.N_NULL)
print('SEKAR_TRAIN_LOOPS x STEPS_PER_LOOP =', P.SEKAR_TRAIN_LOOPS, 'x', P.SEKAR_STEPS_PER_LOOP)

## Optional: quick mechanical smoke test
Run this first if you've changed anything — it finishes in well under a minute and exercises every code path (all 6 pointwise triggers, Sekar's policy training, 2 null directions) at a tiny scale, writing to a separate directory so it never collides with the real results. Skip straight to the full run below if you haven't touched `task12_program.py`.

In [ ]:
RUN_SMOKE_TEST = True

if RUN_SMOKE_TEST:
    os.environ['TASK12_OUT'] = '/kaggle/working/task12_smoke'
    importlib.reload(P)
    P.set_scale(N_BUDGET=400, FINETUNE_STEPS=30, N_EVAL_TRAJ=4, N_NULL=2,
                SEKAR_TRAIN_LOOPS=4, SEKAR_STEPS_PER_LOOP=20)
    P.run_task('cartpole')
    os.environ['TASK12_OUT'] = '/kaggle/working/task12_results'
    importlib.reload(P)
    print('\nSmoke test passed mechanically -- ready for the full run below.')

## Full run — one cell per task
Each call is independently resumable. If Kaggle's session limit is hit partway through cartpole, just re-run this notebook (or re-run this specific cell) — finished (task, comparator) JSONs are skipped automatically.

Per the design doc: 1 seed per (comparator, task) cell for this first pass; the empirical null (50 random directions) and Sekar et al.'s faithful reimplementation both run every task.

In [ ]:
results = {}
for task in ['cartpole', 'reacher', 'pendulum']:
    results[task] = P.run_task(task, include_null=True, include_sekar=True)

## Final summary — ship rule, per task (not averaged)
Ship rule, fixed before any run (see `task_12_design_decisions.md`): `C_t`'s E^state must be the lowest of every pointwise trigger (`kl`, `recon`, `ema_recon`, `ensemble_disagreement`, `random`), **and** `C_t`'s improvement over random acquisition must exceed the 50-random-direction null's spread at z > 2. Sekar et al.'s result is reported for reference — a different mechanism class (a trained policy, not a pointwise trigger) — and is **not** part of the pass/fail bar itself.

In [ ]:
import json

print(f"{'task':<10}  {'ship':<8}  {'beats_baselines':<16}  {'null z':<8}")
for task, r in results.items():
    print(f"{task:<10}  {('SHIP' if r.get('ship') else 'NO-SHIP'):<8}  "
          f"{str(r.get('beats_baselines')):<16}  {r.get('null', {}).get('z', float('nan')):<8.2f}")

with open('/kaggle/working/task12_final_summary.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nWrote /kaggle/working/task12_final_summary.json -- download this plus task12_results/ '
      'to bring the results back into the main project.')